## RAG pipeline

In [1]:
import os
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from pathlib import Path

c:\Users\USER\miniconda3\envs\denseless\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Data ingestion

#### Ingestion

In [2]:
def process_files(path: str):
    """Process all files in the folder"""
    
    docs = []
    # Find all files in path
    path = Path(path)
    files = list(path.glob("**/*.pdf"))
    print(f'Found {len(files)} files to process')
    
    # Process each one
    for file in files:
        print(f'Processing: {file.name}')
        try: 
            loader = PyMuPDFLoader(str(file))
            doc = loader.load()
            print(type(doc))
            
            for page in doc:
                page.metadata['source_file'] = file.name
                page.metadata['file_type'] = 'pdf'
            
            docs.extend(doc)
            print(f'Loaded {len(doc)} pages')
            
        except Exception as e:
            print(f'Error: {e}')
            
    print(f'\nTotal documents loaded: {len(docs)}')
    return docs            
    

In [3]:
all_pdfs = process_files("../data/pdfs")

Found 5 files to process
Processing: CSC 315Week OneTTTT2.pdf
<class 'list'>
Loaded 11 pages
Processing: CSC315 Number System, Codes and Addressing Modes (2).pdf
<class 'list'>
Loaded 29 pages
Processing: Interconnection Structures.pdf
<class 'list'>
Loaded 10 pages
Processing: Number System, Codes and Addressing Modes.pdf
<class 'list'>
Loaded 26 pages
Processing: Week 1- Fundamentals.pdf
<class 'list'>
Loaded 19 pages

Total documents loaded: 95


#### Chunking

In [4]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n","\n"," ",""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f'Split {len(documents)} documents into {len(split_docs)} chunks')
    
    # Show an example 
    if split_docs:
        print("\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
        
    return split_docs

In [5]:
chunks = split_documents(all_pdfs)
chunks

Split 95 documents into 153 chunks

Example chunk:
Content: 1 
 
CSC 315 
: COMPUTER ARCHITECTURE AND ORGANIZATION I  
 
Week One: Lesson One 
 
 Introduction:  
 
• The concept of computer architecture and organization.  
•    Fundamental building blocks (Str...
Metadata: {'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2024-01-17T13:25:18+01:00', 'source': '..\\data\\pdfs\\CSC 315Week OneTTTT2.pdf', 'file_path': '..\\data\\pdfs\\CSC 315Week OneTTTT2.pdf', 'total_pages': 11, 'format': 'PDF 1.7', 'title': '', 'author': 'DR  MONDAY', 'subject': '', 'keywords': '', 'moddate': '2024-01-17T13:25:18+01:00', 'trapped': '', 'modDate': "D:20240117132518+01'00'", 'creationDate': "D:20240117132518+01'00'", 'page': 0, 'source_file': 'CSC 315Week OneTTTT2.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2024-01-17T13:25:18+01:00', 'source': '..\\data\\pdfs\\CSC 315Week OneTTTT2.pdf', 'file_path': '..\\data\\pdfs\\CSC 315Week OneTTTT2.pdf', 'total_pages': 11, 'format': 'PDF 1.7', 'title': '', 'author': 'DR  MONDAY', 'subject': '', 'keywords': '', 'moddate': '2024-01-17T13:25:18+01:00', 'trapped': '', 'modDate': "D:20240117132518+01'00'", 'creationDate': "D:20240117132518+01'00'", 'page': 0, 'source_file': 'CSC 315Week OneTTTT2.pdf', 'file_type': 'pdf'}, page_content='1 \n \nCSC 315 \n: COMPUTER ARCHITECTURE AND ORGANIZATION I  \n \nWeek One: Lesson One \n \n Introduction:  \n \n• The concept of computer architecture and organization.  \n•    Fundamental building blocks (Structure and Function).  \n•   The Von Neumann Architecture.'),
 Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2024-01-17T13:25:18+01:00', 'source': '.

#### Embedding

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer   # Embedding model
import chromadb
from chromadb.config import Settings
import uuid # id for every record in db
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity  # For searching

In [7]:
class EmbeddingManager():
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize embedding manager
        
        Args:
            model_name: HuggingFace model name for embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()
        
    def _load_model(self):
        """Load SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise 
    
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of array embeddings with shape (len(texts), embedding_dim)
        """
        
        if not self.model:
            raise ValueError("model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape {embeddings.shape}")
        return embeddings
        

In [8]:
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6993.22it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384


#### Vector database

In [9]:
class VectorStore:
    """Manages document vector embeddings in a chromadb vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store 
        
        Args:
            collection_name: Name of the chromadb collection
            persist_directory: Directory to access vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None 
        self._initialize_store()
    
    def _initialize_store(self):
        """Initialize chromadb and collection"""
        try:
            # Create persistent chroma db client
            os.makedirs(self.persist_directory, exist_ok=True) ## Create the dir
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata={"description": "pdf document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
           print(f"Error initializing vector store: {e}")
           raise
       
    def add_documents(self, documents: List[Any], embeddings: np.array):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f'Adding {len(documents)} documents to vector store...')
        
        # Prep data for chroma
        ids = []
        metadata = []
        documents_text = []
        embedding_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate uuid 
            doc_id = f'doc_{uuid.uuid4().hex[:8]}_{i}'
            ids.append(doc_id)
            
            # Prep metadata
            metadatum = dict(doc.metadata)
            metadatum['doc_index'] = i
            metadatum['content_length'] = len(doc.page_content)
            metadata.append(metadatum)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embedding_list.append(embedding.tolist())
            
        try:
            self.collection.add(
                ids=ids,
                embeddings=embedding_list,
                metadatas=metadata,
                documents=documents_text,
            )
            print(f'Successfully added {len(documents)} documents to vector store')
            print(f'Total documents in collection: {self.collection.count()}')
            
        except Exception as e:
            print(f'Error adding documents to vector store: {e}')
            raise 

In [10]:
vectorstore = VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 459


#### Embedding and storing

In [11]:
# Convert text to embeddings
texts = [doc.page_content for doc in chunks]
texts

['1 \n \nCSC 315 \n: COMPUTER ARCHITECTURE AND ORGANIZATION I  \n \nWeek One: Lesson One \n \n Introduction:  \n \n• The concept of computer architecture and organization.  \n•    Fundamental building blocks (Structure and Function).  \n•   The Von Neumann Architecture.',
 '2 \n \n \nIn describing computers, a distinction is often made between computer \narchitecture and computer organization. Although it is difficult to give precise \ndefinitions for these terms, a consensus exists about the general areas covered \nby each. For example, (Vranesic and Thurber, 1980: Siewiorek, Bell and Newell, \n1982: Bell,  Mudge  and McNamara 1978). \n \nComputer architecture refers to those attributes of a system visible to a pro- \ngrammer or, put another way, those attributes that have a direct impact on the \nlogical execution of a program.  \nA term that is often used interchangeably with computer architecture is instruction set \narchitecture (ISA). The ISA defines instruction formats, instruct

In [12]:
texts = [doc.page_content for doc in chunks]
texts

['1 \n \nCSC 315 \n: COMPUTER ARCHITECTURE AND ORGANIZATION I  \n \nWeek One: Lesson One \n \n Introduction:  \n \n• The concept of computer architecture and organization.  \n•    Fundamental building blocks (Structure and Function).  \n•   The Von Neumann Architecture.',
 '2 \n \n \nIn describing computers, a distinction is often made between computer \narchitecture and computer organization. Although it is difficult to give precise \ndefinitions for these terms, a consensus exists about the general areas covered \nby each. For example, (Vranesic and Thurber, 1980: Siewiorek, Bell and Newell, \n1982: Bell,  Mudge  and McNamara 1978). \n \nComputer architecture refers to those attributes of a system visible to a pro- \ngrammer or, put another way, those attributes that have a direct impact on the \nlogical execution of a program.  \nA term that is often used interchangeably with computer architecture is instruction set \narchitecture (ISA). The ISA defines instruction formats, instruct

In [13]:
embeddings = embedding_manager.generate_embeddings(texts)
embeddings

Generating embeddings for 153 texts...


Batches: 100%|██████████| 5/5 [00:05<00:00,  1.04s/it]

Generated embeddings with shape (153, 384)


array([[-0.06275751,  0.0088083 , -0.09585551, ...,  0.02913435,
        -0.04609421,  0.00719557],
       [ 0.03711362,  0.04314414, -0.04465407, ...,  0.05059597,
         0.02356858, -0.08289731],
       [ 0.05038413,  0.04010183, -0.06782141, ...,  0.03273155,
         0.00707423, -0.06714777],
       ...,
       [ 0.00489479, -0.03018617, -0.01363951, ..., -0.00112102,
        -0.02942379,  0.02013423],
       [ 0.0104589 , -0.04190646, -0.0187605 , ...,  0.04448584,
        -0.01358809,  0.04445344],
       [-0.01577421, -0.0513883 , -0.04324041, ...,  0.08182956,
         0.00479514,  0.02778831]], shape=(153, 384), dtype=float32)

In [14]:
vectorstore.add_documents(chunks, embeddings)

Adding 153 documents to vector store...
Successfully added 153 documents to vector store
Total documents in collection: 612


### Retrieval pipeline

In [15]:
class RAGRetriever:
    """Handles query based retrieval from vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing embeddings
            embedding_manager: Manager for creating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
        
    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata 
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top k: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # Process results
            retrieved_doc = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (Chromadb uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_doc.append(
                            {
                                'id': doc_id,
                                'content': document,
                                'metadata': metadata,
                                'similarity_score': similarity_score,
                                'distance': distance,
                                'rank': i+1,
                            }
                        )
                        
                print(f"Retrieved {len(retrieved_doc)} documents (after filtering)")
            else:
                print("no documents found")
                
            return retrieved_doc
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

In [16]:
rag_retriever = RAGRetriever(vectorstore, embedding_manager)
rag_retriever

In [17]:
rag_retriever.retrieve("What are interconnection structures and their types")

Retrieving documents for query: 'What are interconnection structures and their types'
Top k: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 47.70it/s]

Generated embeddings with shape (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_2b7d651c_71',
  'content': '1 \n \nCOMPUTER ARCHITECTURE & ORGANIZATION LECTURE  \n(WEEK 2) \n \nInterconnection Structures:  \n• Bus structures: Single bus, Multiple bus  \n• Instruction set Architecture: CISC, RISC  \n• Instruction and instruction sequencing:  \n• Register transfer notation, Instruction types. \n \nInterconnection structures \n• Computers has three main or basic modules: Processor, Memory and I/O \no The three basic modules must communicate with each other, therefore there \nmust be paths for connecting these modules for them to communicate with \neach other. \nThe collection of paths connecting the various modules is called the \ninterconnection structure. The design of this structure will depend on the \nexchanges that must be made among modules. \n \n■ Memory:  \n Typically, a memory module will consist of N words of equal length.  \n Each word is assigned a unique numerical address (0, 1,  N-1).  \n A word of data can be read from or written into the

#### LLM integration

In [18]:
from langchain_ollama import ChatOllama

In [19]:
llm = ChatOllama(model="llama3.2:3b")

# simple rag function
def rag_simple(query, retriever, llm, top_k=3):
    # Retrieve context
    results = retriever.retrieve(query, top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer question"
    
    # Generate answer
    prompt = f"""
    Use the following context to answer the question concisely.
    
    Context:
    {context}
    
    Question: {query}
    
    Answer: 
    """
    response = llm.invoke([prompt.format(context=context, query=query)])
    return response.content

In [20]:
answer = rag_simple("Define interconnection structures", rag_retriever, llm)
answer

Retrieving documents for query: 'Define interconnection structures'
Top k: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 57.40it/s]

Generated embeddings with shape (1, 384)
Retrieved 3 documents (after filtering)


'Interconnection structures refer to the paths connecting the basic modules of a computer system, which include the Processor, Memory, and I/O. These paths enable communication and exchange between the modules. The design of an interconnection structure depends on the exchanges that must be made among the modules, and can take various forms such as bus structures (single or multiple bus) that facilitate data transfer.'

In [21]:
# Advanced RAG function
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prep context and sources
    context = '\n\n'.join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Üse the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence,
    }
    
    if return_context:
        output['context'] = context
    
    return output

In [22]:
result = rag_advanced("What is Computer organization and computer architecture?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context preview:", result['context'][:300])

Retrieving documents for query: 'What is Computer organization and computer architecture?'
Top k: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 61.45it/s]

Generated embeddings with shape (1, 384)
Retrieved 3 documents (after filtering)


Answer: Based on the provided context, here is a concise answer:

Computer Organization refers to the design and structure of a computer's internal components and their interactions. It determines how the hardware elements work together to perform tasks.

Computer Architecture refers to the overall design and configuration of a computer's hardware and software components. It outlines the functional specifications of a system, including the organization, interconnection, and behavior of its parts.

In essence, Computer Organization is about how things are organized, while Computer Architecture is about what things are used.
Sources: [{'source': 'CSC 315Week OneTTTT2.pdf', 'page': 2, 'score': 0.5848396420478821, 'preview': '3 \n \nThis architecture was first introduced in 1970 and with a  few enhancements, has \nsurvived to this day as the architecture of IBM’s mainframe product line. \n \nIn  microcomputers, the relationship between architecture and organization is very \nclose. Changes